# Emotion Prediction on DistilRoBERTa
[michellejieli/emotion_text_classifier](https://huggingface.co/michellejieli/emotion_text_classifier)

In [1]:
import os
import pandas as pd
import torch
import wandb
from datasets import load_dataset, Dataset
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, pipeline
from huggingface_hub import login
from peft import LoraConfig, get_peft_model

In [2]:
# prevent env load failed
%load_ext dotenv
%dotenv

In [3]:
login(token=os.environ.get("HF_TOKEN", ""), add_to_git_credential=True)
wandb.login(key=os.environ.get("WANDB_API_KEY", ""), relogin=True)

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (cache).
Your token has been saved to /home/user/.cache/huggingface/token
Login successful


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/user/.netrc


True

In [4]:
# # Initialize Wandb
# wandb_config = {
#     "base_model": "michellejieli/emotion_text_classifier",
# }
# wandb.init(
#     job_type="fine-tuning",
#     config=wandb_config,
#     project="emotion-chat-bot-ncu",
#     group="candidate_generation",
#     mode="online",
#     # resume="auto"
# )

# data preprocessing
把 label 向後平移一個

In [5]:
# Model from Hugging Face hub
base_model = "michellejieli/emotion_text_classifier"

# Saved fine-tuned model 
new_model = "test_model"

In [6]:
def preprocessing(data):
    data = data.rename_column("utterance", "text")
    data = data.rename_column("emotion", "label")
    data = data.remove_columns("turn_type")
    # data = data.remove_columns(["dialog_id", "turn_type"])
    return data

def shifting_test(data):
    df = data.to_pandas()
    df["label"] = df.groupby('dialog_id')["label"].shift(-1)
    df.dropna(inplace = True)
    df["label"]  = df["label"].astype(int)
    print(df.head(20))
    modified_dataset = Dataset.from_pandas(df)
    data = modified_dataset
    return data

def shifting_train(data):
    df = data["train"].to_pandas()
    df["label"] = df.groupby('dialog_id')["label"].shift(-1)
    df.dropna(inplace = True)
    df["label"]  = df["label"].astype(int)
    modified_dataset = Dataset.from_pandas(df)
    data["train"] = modified_dataset
    return data

def shift_labels(dataset):
    df = dataset.to_pandas()
    df["label"] = df.groupby('dialog_id')["label"].shift(-1)
    df.dropna(inplace = True)
    df["label"]  = df["label"].astype(int)
    dataset = Dataset.from_pandas(df)
    dataset = dataset.remove_columns("dialog_id")
    return dataset


def shift_all(data):
    data["train"] = shift_labels(data["train"])
    data["validation"] = shift_labels(data["validation"])
    data["test"] = shift_labels(data["test"])
    # data = data.remove_columns("__index_level_0__")
    return data

In [7]:
data_name = "benjaminbeilharz/better_daily_dialog"
# data_raw = load_dataset(data_name, num_proc=16)
data_raw = load_dataset(data_name, num_proc=16)
data_raw = preprocessing(data_raw)
data_raw



DatasetDict({
    train: Dataset({
        features: ['dialog_id', 'text', 'label'],
        num_rows: 87170
    })
    validation: Dataset({
        features: ['dialog_id', 'text', 'label'],
        num_rows: 8069
    })
    test: Dataset({
        features: ['dialog_id', 'text', 'label'],
        num_rows: 7740
    })
})

In [8]:
# test_data = load_dataset(data_name, split='test', num_proc=8)
# test_data = preprocessing(data_raw["test"])
test_data = shifting_test(data_raw["test"])
test_data


    dialog_id                                               text  label
0           0               Hey man , you wanna buy some weed ?       6
1           0                                       Some what ?       0
2           0   Weed ! You know ? Pot , Ganja , Mary Jane som...      0
3           0                            Oh , umm , no thanks .       0
4           0   I also have blow if you prefer to do a few li...      0
5           0                           No , I am ok , really .       0
6           0   Come on man ! I even got dope and acid ! Try ...      0
7           0   Do you really have all of these drugs ? Where...      0
8           0   I got my connections ! Just tell me what you ...      0
9           0              Sounds good ! Let ’ s see , I want .       3
10          0                                            Yeah ?       0
12          1            The taxi drivers are on strike again .       0
13          1                                        What for ? 

Dataset({
    features: ['dialog_id', 'text', 'label', '__index_level_0__'],
    num_rows: 6740
})

In [9]:
data = data_raw

In [10]:
# 获取原始训练集
train_dataset = data["train"]

# 计算提取的前10%的数量
# num_samples = int(len(train_dataset) * 0.01)

# 提取前10%的数据
train_10_percent = train_dataset.select(range(1000))

# 更新DatasetDict中的训练集
data["train"] = train_10_percent

# 打印修改后的DatasetDict
print(data)


DatasetDict({
    train: Dataset({
        features: ['dialog_id', 'text', 'label'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['dialog_id', 'text', 'label'],
        num_rows: 8069
    })
    test: Dataset({
        features: ['dialog_id', 'text', 'label'],
        num_rows: 7740
    })
})


In [11]:
# df = data["train"].to_pandas()
# df["label"] = df.groupby('dialog_id')["label"].shift(-1)
# print(df.head(20))

In [12]:
# df.dropna(inplace = True)
# df["label"]  = df["label"].astype(int)
# print(df.head(20))

In [13]:
shift_all(data)

DatasetDict({
    train: Dataset({
        features: ['text', 'label', '__index_level_0__'],
        num_rows: 858
    })
    validation: Dataset({
        features: ['text', 'label', '__index_level_0__'],
        num_rows: 7069
    })
    test: Dataset({
        features: ['text', 'label', '__index_level_0__'],
        num_rows: 6740
    })
})

In [14]:
# shifting_train(data)

In [15]:
data["train"]["label"][0:20]

[0, 0, 0, 0, 0, 4, 4, 4, 4, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0]

In [16]:
data["validation"]["label"][0:20]

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

In [17]:
tokenizer = AutoTokenizer.from_pretrained(base_model)

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

In [18]:
emotions = data
print(tokenize(emotions["train"][:2]))

{'input_ids': [[0, 34673, 2156, 2488, 2156, 141, 59, 164, 13, 10, 367, 16328, 71, 3630, 17487, 1437, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [19]:
#hide_input
#not very sure what did he do here
tokens2ids = list(zip(tokenizer.all_special_tokens, tokenizer.all_special_ids))
data = sorted(tokens2ids, key=lambda x: x[-1])
df = pd.DataFrame(data, columns=["Special Token", "Special Token ID"])
df.T

,0,1,2,3,4
Special Token,<s>,<pad>,</s>,<unk>,<mask>
Special Token ID,0,1,2,3,50264


In [20]:
 # hide_output
emotions_encoded = emotions.map(tokenize, batched=True, batch_size=None)
emotions_encoded = emotions_encoded.remove_columns(['__index_level_0__'])
print(emotions_encoded.column_names)

Map:   0%|          | 0/858 [00:00<?, ? examples/s]

Map:   0%|          | 0/7069 [00:00<?, ? examples/s]

Map:   0%|          | 0/6740 [00:00<?, ? examples/s]

{'train': ['text', 'label', 'input_ids', 'attention_mask'], 'validation': ['text', 'label', 'input_ids', 'attention_mask'], 'test': ['text', 'label', 'input_ids', 'attention_mask']}


In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# hide_output
num_labels = 7
id2label = {
    0: "neutral",
    1: "anger",
    2: "disgust",
    3: "fear",
    4: "happiness",
    5: "sadness",
    6: "surprise"
}

label2id = {
    "neutral": 0,
    "anger": 1,
    "disgust": 2,
    "fear": 3,
    "happiness": 4,
    "sadness": 5,
    "surprise": 6
}

model = AutoModelForSequenceClassification.from_pretrained(base_model, num_labels=num_labels, id2label=id2label, label2id=label2id)

In [22]:
# lora_config = LoraConfig(
#     lora_alpha=16,
#     lora_dropout=0.1,
#     r=8,
#     bias="none",
#     task_type="SEQ_CLS",
#     use_rslora = True
# )

In [23]:
# peft_model = get_peft_model(model, lora_config)

In [24]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1 = f1_score(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    wandb.log({"eval_f1": f1})
    return {"accuracy": acc, "f1": f1}

## Sweep
HF 給的東西實在是看不懂，改用[這個](https://wandb.ai/matt24/vit-snacks-sweeps/reports/Hyperparameter-Search-with-W-B-Sweeps-for-Hugging-Face-Transformer-Models--VmlldzoyMTUxNTg0)

再改用[這個](https://community.wandb.ai/t/what-is-the-official-way-to-run-a-wandb-sweep-with-hugging-face-hf-transformers/4668/6)

參考[這個](https://docs.wandb.ai/tutorials/xgboost_sweeps)

In [25]:
sweep_config = {
    "method": "random",
    "name": "disaster-sweep",
    "metric": {
        "goal": "maximize",
        "name": "eval_f1"
    },
    "parameters": {
        "epochs": {
            "value": 2
        },
        "batch_size": {
            "values": [8, 16, 32, 64]
        },
        "learning_rate": {
            "values": [0.005, 0.0001, 0.00005]
        },
        "weight_decay": {
            "values": [0.0001, 0.1]
        },
        "lora_r": {
            "values": [8, 16, 64, 128, 256]
        },
        "lora_alpha": {
            "values": [16, 32, 64]
        },
        "lora_dropout": {
            "values": [0.0, 0.1, 0.2]
        }
    }
}


In [26]:
def train(config=None):
  with wandb.init(config=config):
    # set sweep configuration
    config = wandb.config


    # set training arguments
    training_args = TrainingArguments(
    output_dir="./sweeps",
	  report_to='wandb',  # Turn on Weights & Biases logging
        num_train_epochs=config.epochs,
        learning_rate=config.learning_rate,
        weight_decay=config.weight_decay,
        per_device_train_batch_size=config.batch_size,
        per_device_eval_batch_size=config.batch_size,
        # per_device_eval_batch_size=16,
        save_strategy='epoch',
        evaluation_strategy='epoch',
        logging_strategy='epoch',
        load_best_model_at_end=True,
        remove_unused_columns=False,
        fp16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": True}
    )

    lora_config = LoraConfig(
        r=config.lora_r,
        lora_alpha=config.lora_alpha,
        lora_dropout=config.lora_dropout,
        bias="none",
        task_type="SEQ_CLS",
        use_rslora = True
    )
    
    peft_model = get_peft_model(model, lora_config)

    trainer = Trainer(
        model=peft_model,
        args=training_args,
        train_dataset=emotions_encoded["train"],
        eval_dataset=emotions_encoded["validation"],
        compute_metrics=compute_metrics,
        # tokenizer=tokenizer,
        # model_init=model_init,
        # data_collator=data_collator,
    )


    # start training loop
    trainer.train()


In [27]:
sweep_id = wandb.sweep(sweep_config, project='test-sweeps')

Create sweep with ID: x0ca9vef
Sweep URL: https://wandb.ai/yangyx30678/test-sweeps/sweeps/x0ca9vef


In [28]:
wandb.agent(sweep_id, train, count=10)

wandb: Agent Starting Run: vklyd6bw with config:
wandb: 	batch_size: 8
wandb: 	epochs: 2
wandb: 	learning_rate: 0.0001
wandb: 	lora_alpha: 32
wandb: 	lora_dropout: 0.1
wandb: 	lora_r: 8
wandb: 	weight_decay: 0.0001
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: yangyx30678. Use `wandb login --relogin` to force relogin


/home/user/.cache/pypoetry/virtualenvs/chat-bot-20tW9agt-py3.11/lib/python3.11/site-packages/accelerate/accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(
wandb: WARNING Config item 'learning_rate' was locked by 'sweep' (ignored update).
wandb: WARNING Config item 'weight_decay' was locked by 'sweep' (ignored update).


  0%|          | 0/216 [00:00<?, ?it/s]

/home/user/.cache/pypoetry/virtualenvs/chat-bot-20tW9agt-py3.11/lib/python3.11/site-packages/torch/utils/checkpoint.py:91: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


{'loss': 1.0933, 'grad_norm': 13.52354907989502, 'learning_rate': 5e-05, 'epoch': 1.0}


...

{'eval_loss': 0.6995945572853088, 'eval_accuracy': 0.8036497382939596, 'eval_f1': 0.795796242547334, 'eval_runtime': 16.8515, 'eval_samples_per_second': 419.487, 'eval_steps_per_second': 13.115, 'epoch': 1.0}
{'loss': 0.9578, 'grad_norm': 5.7337188720703125, 'learning_rate': 0.0, 'epoch': 2.0}


  0%|          | 0/221 [00:00<?, ?it/s]

wandb: Ctrl + C detected. Stopping sweep.


Checkpoint destination directory ./sweeps/checkpoint-54 already exists and is non-empty. Saving will proceed but saved results may be invalid.


{'eval_loss': 0.6099016070365906, 'eval_accuracy': 0.8400056585089829, 'eval_f1': 0.8108534479963309, 'eval_runtime': 16.6182, 'eval_samples_per_second': 425.376, 'eval_steps_per_second': 13.299, 'epoch': 2.0}
{'train_runtime': 48.4583, 'train_samples_per_second': 35.412, 'train_steps_per_second': 1.114, 'train_loss': 1.440897093878852, 'epoch': 2.0}


## Train

In [ ]:
batch_size = 64
logging_steps = len(emotions_encoded["train"]) // batch_size

In [ ]:
training_args = TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=10,
    load_best_model_at_end = True,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=1,
    optim="paged_adamw_32bit",
    save_steps=100,
    save_strategy = "epoch",
    logging_steps=logging_steps,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=False,
    bf16=False,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="constant",
    report_to=["wandb"],
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": True},
    evaluation_strategy="epoch",
    log_level="error",
    overwrite_output_dir=True
)

In [ ]:
wandb.config["trainer_arguments"] = training_args.to_dict()

In [ ]:
trainer = Trainer(model=peft_model, args=training_args,
                  compute_metrics=compute_metrics,
                  train_dataset=emotions_encoded["train"],
                  eval_dataset=emotions_encoded["validation"],
                  tokenizer=tokenizer)
# trainer.train(resume_from_checkpoint=True);
# trainer.model.save_pretrained(new_model)


In [ ]:
# trainer.args.num_train_epochs = trainer.args.num_train_epochs + 5

In [ ]:
trainer.train()

In [ ]:
# model = AutoModelForSequenceClassification.from_pretrained(base_model, num_labels=num_labels, id2label=id2label, label2id=label2id)
# trainer1 = Trainer(model=model, args=training_args,
#                   compute_metrics=compute_metrics,
#                   train_dataset=emotions_encoded["train"],
#                   eval_dataset=emotions_encoded["validation"],
#                   tokenizer=tokenizer)

In [ ]:
# import matplotlib.pyplot as plt
# f1_scores = []
# accuracies = []

# for i in range(2):
#     model = AutoModelForSequenceClassification.from_pretrained(new_model, num_labels=num_labels, id2label=id2label, label2id=label2id)
#     trainer1 = Trainer(model=model, args=training_args,
#               compute_metrics=compute_metrics,
#               train_dataset=emotions_encoded["train"],
#               eval_dataset=emotions_encoded["validation"],
#               tokenizer=tokenizer)
#     trainer1.train()
#     # print("finished an epoch")
#     trainer1.model.save_pretrained(new_model)
#     # print("saved")
#     # classifier_model = AutoModelForSequenceClassification.from_pretrained(new_model, num_labels=num_labels, id2label=id2label, label2id=label2id)
#     classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=0)
#     def predict(row):
#         text = row['text']
#         true_label = row['label']
#         predicted_result = classifier(text)[0]
#         predicted_label = label2id[predicted_result["label"]]
#         return {"predicted_label": predicted_label, "true_label": true_label}

#     predictions = test_data.map(predict)
#     true_labels = [p["true_label"] for p in predictions]
#     predicted_labels = [p["predicted_label"] for p in predictions]

#     f1 = f1_score(true_labels, predicted_labels, average='weighted')
#     accuracy = accuracy_score(true_labels, predicted_labels)

#     print("F1-score:", f1, )
#     print("Accuracy:", accuracy)
    
#     f1_scores.append(f1)
#     accuracies.append(accuracy)

In [ ]:
# # 绘制 F1-score 和 Accuracy 的变化趋势
# plt.figure(figsize=(12, 6))

# plt.subplot(1, 2, 1)
# plt.plot(range(1, len(f1_scores) + 1), f1_scores, marker='o')
# plt.title('F1-score over iterations')
# plt.xlabel('Iteration')
# plt.ylabel('F1-score')

# plt.subplot(1, 2, 2)
# plt.plot(range(1, len(accuracies) + 1), accuracies, marker='o')
# plt.title('Accuracy over iterations')
# plt.xlabel('Iteration')
# plt.ylabel('Accuracy')

In [ ]:
wandb.finish()

# Inference
I use first five percent of train data to predict

In [ ]:
classifier_model = AutoModelForSequenceClassification.from_pretrained(new_model, num_labels=num_labels, id2label=id2label, label2id=label2id)

In [ ]:
classifier = pipeline("sentiment-analysis", model=classifier_model, tokenizer=tokenizer, device=0)
data_name = "benjaminbeilharz/better_daily_dialog"
data = load_dataset(data_name, split='test', num_proc=8)
data = preprocessing(data)

In [ ]:
data[0:10]

In [ ]:
data = shifting_test(data)

In [ ]:
data[0:10]

In [ ]:

def predict(row):
    text = row['text']
    true_label = row['label']
    predicted_result = classifier(text)[0]
    predicted_label = label2id[predicted_result["label"]]

    # print(f"Predicted: {predicted_label}, True: {true_label},        ##Text: {text}")
    return {"predicted_label": predicted_label, "true_label": true_label}
predictions = data.map(predict)
true_labels = [p["true_label"] for p in predictions]
predicted_labels = [p["predicted_label"] for p in predictions]

f1 = f1_score(true_labels, predicted_labels, average='weighted')
accuracy = accuracy_score(true_labels, predicted_labels)

print("F1-score:", f1, )
print("Accuracy:", accuracy)